In [ ]:

%load_ext aiida
%aiida
import urllib.parse as urlparse

import ipywidgets as ipw
from IPython.display import display, clear_output,FileLink,HTML
import tempfile
import subprocess
import shutil
import os
import threading
import time
from tornado.ioloop import IOLoop

from aiida import orm
from aiidalab_widgets_base import viewer

from empasiesta_tools.widgets import comments, obsolete

<IPython.core.display.Javascript object>

In [ ]:
pk = urlparse.parse_qs(urlparse.urlsplit(jupyter_notebook_url).query)["pk"][0]
workcalc = orm.load_node(pk)
init_structure = workcalc.inputs.structure
try:
    opt_structure = workcalc.outputs.output_structure
except:
    opt_structure = None

## Equilibrium / input geometry

In [3]:
description = ipw.HTML(value=f'<b style="color:blue;">{workcalc.description}</b><br>')


# Your structures
opt_structure = opt_structure   # can be None
view_fn = viewer                # your viewer function

# ---------------------------------------------------------
# 1. Create selection widget
# ---------------------------------------------------------
options = ["initial"]
if opt_structure is not None:
    options.append("optimized")

toggle = ipw.ToggleButtons(
    options=options,
    description="Structure:",
    disabled=False,
    button_style='',
)

# Output area where the viewer will be displayed
out = ipw.Output()

# ---------------------------------------------------------
# 2. Callback to update the display
# ---------------------------------------------------------
def update_view(change=None):
    with out:
        clear_output()
        if toggle.value == "optimized" and opt_structure is not None:
            structure_to_show = opt_structure
            desc_text = "<b style='color:blue;'>Optimized structure</b>"
        else:
            structure_to_show = init_structure
            desc_text = "<b style='color:blue;'>Initial structure</b>"
        
        desc = ipw.HTML(value=desc_text)
        display(desc, view_fn(structure_to_show))

# ---------------------------------------------------------
# 3. Attach callback
# ---------------------------------------------------------
toggle.observe(update_view, names="value")

# ---------------------------------------------------------
# 4. Initial display
# ---------------------------------------------------------
display(toggle, out)
update_view()


ToggleButtons(description='Structure:', options=('initial',), value='initial')

Output()

# Bands

In [5]:
try:
    bands = workcalc.outputs.bands
    display(viewer(bands))
except:
    pass

## Comments

In [6]:
comments_widget = comments.CommentsWidget(workchain=pk)
display(comments_widget)

CommentsWidget(children=(VBox(children=(Output(), Textarea(value='', layout=Layout(width='60%')), Button(descr…

## Mark calculation as obsolete 

In [7]:
obsolete = obsolete.ObsoleteWidget(workchain=pk)
display(obsolete)

ObsoleteWidget(children=(VBox(children=(Button(button_style='danger', description='Mark as obsolete', style=Bu…

# Dump files

In [ ]:
def make_download_widget(pk, lifetime=50):
    output = ipw.Output()

    def delete_file(path_abs, label):
        """Remove file and update label."""
        try:
            if os.path.exists(path_abs):
                os.remove(path_abs)
            label.value = (
                '<span style="color:green; font-size:18px; font-weight:bold;">'
                'File deleted ✔'
                '</span>'
            )
        except Exception as e:
            label.value = str(e)

    def update_countdown(label, remaining, path_abs):
        """Update countdown text every second without blocking kernel."""
        if remaining <= 0:
            delete_file(path_abs, label)
            return

        label.value = (
            f'<span style="color:green; font-size:18px; font-weight:bold;">'
            f'After downloading, please wait for auto deletion in {remaining} seconds…'
            f'</span>'
        )

        # schedule next tick in 1 second
        IOLoop.current().call_later(
            1, lambda: update_countdown(label, remaining - 1, path_abs)
        )

    def on_click(b):
        with output:
            output.clear_output()
            print(f"Dumping workchain {pk} …")

            # 1. Temporary dir
            tmpdir = tempfile.mkdtemp()
            dump_dir = os.path.join(tmpdir, f"DumpOf-{pk}")

            # 2. Dump
            subprocess.run(["verdi", "process", "dump", "-p", dump_dir, str(pk)], check=True)

            # 3. Zip
            zip_path_tmp = shutil.make_archive(dump_dir, "zip", dump_dir)

            # 4. Move zip to served directory
            download_dir = "downloads"
            os.makedirs(download_dir, exist_ok=True)

            filename = os.path.basename(zip_path_tmp)
            zip_path_abs = os.path.join(download_dir, filename)
            shutil.copy(zip_path_tmp, zip_path_abs)

            # 5. Download link
            rel_path = f"{download_dir}/{filename}"
            display(HTML(
                f'<a href="{rel_path}" style="font-size:24px; font-weight:bold; color:#0074D9;" download>'
                f'⬇ Download dump file'
                f'</a>'
            ))

            # 6. Countdown label
            countdown_label = ipw.HTML()
            display(countdown_label)

            # Start countdown + delete logic (no threads)
            update_countdown(countdown_label, lifetime, zip_path_abs)

    button = ipw.Button(description="Download files", button_style="success")
    button.on_click(on_click)

    return ipw.VBox([button, output])


In [ ]:
widget = make_download_widget(pk)
display(widget)